In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats

In [ ]:
TASK = Path.cwd().parent
INPUT, OUTPUT = TASK/'input/fields', TASK/'output'
FIELDS = {
    'biological_sciences': 'Biological sciences', 'chemistry': 'Chemistry',
    'computer_science': 'Computer science (AARC)', 'economics': 'Economics',
    'electrical_engineering': 'Electrical engineering', 'history': 'History',
    'management': 'Management', 'mathematics': 'Mathematics',
    'mechanical_engineering': 'Mechanical engineering', 'physics': 'Physics',
    'political_science': 'Political science', 'psychology': 'Psychology', 'sociology': 'Sociology',
}
COLUMNS = ['PersonId', 'CareerAgeZero', 'pubs_adj']
WINDOW, MIN_END_AGE = 4, 4  # keep the Shockley notebook's career-age cutoff

In [ ]:
files = {path.stem.removeprefix('secret_').removesuffix('_trajectories'): path
         for path in INPUT.glob('secret_*_trajectories.csv')}
missing = FIELDS.keys() - files.keys()
if missing:
    raise FileNotFoundError(f'Missing AARC field trajectory exports: {", ".join(sorted(missing))}')

In [ ]:
rows = []
for field, label in FIELDS.items():
    data = pd.read_csv(files[field], usecols=COLUMNS).sort_values(['PersonId', 'CareerAgeZero'])
    if data.duplicated(['PersonId', 'CareerAgeZero']).any():
        raise ValueError(f'{files[field].name}: expected one row per researcher and career year')

    # last complete four-row window, using observed rows as in the draft
    data['total'] = data.groupby('PersonId', sort=False).pubs_adj.rolling(WINDOW).sum().droplevel(0)
    final = data.dropna(subset=['total']).drop_duplicates('PersonId', keep='last')
    eligible = final.loc[final.CareerAgeZero.ge(MIN_END_AGE), 'total']
    positive = eligible.loc[eligible.gt(0) & np.isfinite(eligible)]
    if len(positive) < 2 or positive.nunique() < 2:
        raise ValueError(f'{files[field].name}: insufficient variation in positive four-year totals')

    # standardizing the logs leaves the probability-plot correlation unchanged
    _, (_, _, r) = stats.probplot(np.log2(positive), dist='norm')
    rows.append({'field': field, 'label': label, 'n_eligible': len(eligible),
                 'n_positive': len(positive), 'qq_r': r})

In [ ]:
summary = pd.DataFrame(rows).sort_values(['qq_r', 'field']).reset_index(drop=True)
summary.to_csv(OUTPUT/'aarc_field_lognormality.csv', index=False)
display(summary)